# 📗 청킹 전략과 검색 품질

## 오늘의 목표

- [ ] 기본 RAG가 근거를 놓치는 이유를 설명할 수 있습니다.
- [ ] Fixed·Semantic의 설정을 바꿔 청크 경계를 비교할 수 있습니다.
- [ ] Parent-Child·Sentence Window·Auto-merging으로 검색 단위와 반환 단위를 나눌 수 있습니다.
- [ ] 같은 질문·근거·K로 근거 회수와 반환 글자 수를 비교할 수 있습니다.

## ⏪ 지난 시간 복습

앞 단원에서는 원문을 `Document`로 만들고, 나눈 청크를 임베딩해 `Chroma`에 넣은 뒤 질문과 가까운 청크를 검색했습니다(Document → 분할 → 임베딩 → Chroma → 검색). 오늘은 이 흐름의 **분할과 반환 단위**를 바꿉니다.

## 기본 RAG의 한계와 오늘 바꿀 단계

문서를 한 가지 크기로 자르고 질문과 가까운 청크 K개를 그대로 답변에 넘기는 구성을 **Naive RAG**(기본 RAG)라고 합니다. 이 구성은 두 곳에서 자주 틀립니다.

- **근거가 청크 경계에서 잘립니다.** 도서관 안내를 겹침 없이 80자로 자르면 “…예약이 없으면 한”과 “번에 한해 일주일 연장할 수 있습니다”가 다른 청크가 됩니다(1절).
- **질문 범위와 청크 크기가 맞지 않습니다.** 두 절에 걸친 질문은 작은 청크 하나로 답할 수 없고, 청크를 키우면 필요 없는 문장까지 넘어갑니다.

**Advanced RAG**는 검색 앞뒤에 단계를 두어 이 한계를 고칩니다.

| 단계 | 하는 일 | 다루는 시간 |
|---|---|---|
| 검색 전(Pre-retrieval) | 문서를 나누고 인덱스 만들기, 질문 다듬기 | 오늘: 청킹 전략, RAPTOR / 다음 시간: 질의 변환 |
| 검색 | 질문과 가까운 문서 찾기 | 다음 시간: 하이브리드 검색 |
| 검색 후(Post-retrieval) | 찾은 결과를 답변에 넘기기 전에 손보기 | 그다음 시간: 리랭킹, 컨텍스트 압축 |

오늘은 검색 전 단계인 **인덱싱**(무엇으로 찾고 무엇을 돌려줄지)을 바꿉니다. 시연은 가상 도서관 안내, 따라하기는 고용노동부 유연근무 매뉴얼 PDF입니다.

<img style="background:#fff; max-width:100%; height:auto;" src="images/01_search_context.png" width="960" alt="Fixed·Recursive·Semantic 분할과 세 가지 문맥 확장 전략 비교" />

Fixed·Semantic은 **저장할 청크**를 정합니다. Parent-Child·Sentence Window·Auto-merging은 검색 단위와 확장할 문맥을 준비하고, 검색 뒤 **실제 반환 범위**를 결정합니다.

### 준비

경로 변수와 `read_json`, `save_json`은 앞 단원과 같습니다.

In [ ]:
# 원문을 읽고 Document로 담는 공통 도구를 준비합니다.
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

#### 원문과 결과 파일의 경로

In [ ]:
# 실행할 노트북 폴더를 기준으로 입력 data와 생성 결과 output을 구분합니다.
import sys

material_dir = Path(".")
# 학생용·정답용 모두 실습자료 폴더의 util.py를 가져옵니다.
sys.path.insert(0, str(material_dir.resolve()))

data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)


def read_json(name):
    """data 폴더의 JSON 파일을 목록 또는 딕셔너리로 읽습니다."""
    return json.loads((data_dir / name).read_text(encoding="utf-8"))


def save_json(name, value):
    """처리 결과를 output 폴더에 한글을 유지해 저장합니다."""
    (output_dir / name).write_text(
        json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8"
    )

#### 모델 연결 설정

`.env`에 `OPENAI_API_KEY`를 넣습니다(`.env.example` 참고). 임베딩은 앞 단원과 같은 `text-embedding-3-large` 768차원으로 고정합니다. 모델이 같아야 청킹 방식의 차이만 비교할 수 있습니다.

In [ ]:
# 현재 작업 폴더부터 상위로 .env를 찾아 키를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(".env에 OPENAI_API_KEY를 설정한 뒤 이 셀부터 다시 실행하세요.")

# 문서와 질문을 같은 모델·차원으로 바꿔야 같은 인덱스에서 거리를 비교할 수 있습니다.
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large",
    dimensions=768,  # 문장·청크 하나를 768개 숫자로 표현합니다.
    check_embedding_ctx_length=False,  # 자동 길이 검사와 분할을 끄고 준비한 짧은 청크를 보냅니다.
)

print("모델 연결 설정 완료. 임베딩은 본문·질문을 벡터로 바꿀 때 요청합니다.")

#### 청킹할 본문 읽기

`text`를 각 전략의 분할기에 넣습니다. PDF 입력은 여러 페이지가 이어진 긴 본문이며, 문장이나 검색 청크를 미리 정해 두지 않았습니다. `start`·`end`는 이 본문 문자열에서 세는 위치이며 끝 위치는 포함하지 않습니다.

In [ ]:
# 원문 한 편이 기록 하나이며 text가 청킹할 본문입니다.
demo_records = read_json("demo_docs.json")
print("원문 수:", len(demo_records))
display(pd.DataFrame(demo_records))

In [ ]:
# 첫 원문을 읽고 문장과 조건이 어떻게 이어지는지 봅니다.
print(demo_records[0]["text"])

In [ ]:
# 연속된 PDF 페이지를 연결한 긴 본문을 읽습니다. 청킹은 뒤에서 직접 합니다.
practice_records = read_json("practice_docs.json")
print("원문 수:", len(practice_records))
display(pd.DataFrame(practice_records))

#### 실제 PDF와 실습 입력

인사팀이 “유연근무 신청을 반려할 때 어떤 절차가 필요한가요?”를 찾는 상황입니다. 고용노동부 「일·생활 균형을 위한 유연근무 활용 매뉴얼」(2024년 10월)의 **71페이지를 연결한 긴 본문 4편**을 씁니다. 구간 안에서는 페이지나 짧은 절마다 문서를 끊지 않습니다.

| 입력 ID | 내용 | PDF 페이지 |
|---|---|---|
| hr_flexible_intro | 유연근무 개념과 도입 | 9~20 |
| hr_staggered_work | 시차출퇴근 | 25~42 |
| hr_selective_work | 선택근무 | 49~66 |
| hr_remote_work | 재택·원격근무 | 71~93 |

**PDF 페이지 추출 → 반복 머리말 정리 → 연속 페이지 연결 → 전략별 청킹** 순서입니다. `practice_docs.json`에는 같은 PDF에서 반복 머리말과 줄 끝 공백을 정리하고, 연속 페이지의 전체 본문을 연결해 두었습니다. 아래 코드는 원본 PDF 한 쪽을 읽는 예시이며, 청킹에는 준비된 긴 본문의 `text`를 사용합니다.

[원본 PDF](data/originals/MOEL_flexible_work_manual_2024.pdf) · [PyPDFLoader 문서](https://reference.langchain.com/python/langchain-community/document_loaders/pdf/PyPDFLoader)

<img style="background:#fff; max-width:100%; height:auto;" src="images/pdf_page18.png" width="520" alt="PDF 18쪽의 신청과 승인 절차 원문" />

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = data_dir / "originals" / "MOEL_flexible_work_manual_2024.pdf"
# page 모드는 페이지별 Document를 만듭니다. 이는 아직 검색용 청크가 아닙니다.
pdf_pages = PyPDFLoader(str(pdf_path), mode="page").load()
page_number = 18
print("PDF 페이지:", page_number)
# 뷰어 쪽 번호는 1부터, 파이썬 리스트는 0부터 세므로 1을 뺍니다.
print(pdf_pages[page_number - 1].page_content)

`page_spans`는 긴 본문의 문자 위치와 PDF 페이지를 연결하는 출처 정보입니다. 청크 경계는 이후 각 청킹 전략으로 정합니다.

PDF에서 추출한 제목의 띄어쓰기나 표의 행·열 순서는 원본과 다를 수 있습니다. 숫자와 조건은 원본 PDF와 함께 읽으세요.

#### 원문을 LangChain Document로 바꾸기

`text`는 `page_content`에, 원문 ID·제목·위치는 `metadata`에 담습니다. 아직 자르지 않습니다.

In [ ]:
def make_documents(records):
    """원문 기록마다 본문과 출처(ID·제목·위치)를 담은 LangChain Document를 만듭니다."""
    return [
        Document(
            page_content=record["text"],
            # 분할한 뒤에도 source_id로 청킹 전 본문을 다시 찾습니다.
            metadata={
                "source_id": record["doc_id"],
                "title": record["title"],
                "url": record["url"],
                # 원문 전체 구간을 먼저 기록하고, 분할 뒤에는 각 청크의 구간으로 갱신합니다.
                "start_index": 0,
                "end_index": len(record["text"]),
            },
        )
        for record in records
    ]

#### 출처를 유지한 원문 Document 만들기

In [ ]:
# 검색에는 Document를 쓰고, 문맥 확장에는 ID로 조회할 원문 딕셔너리를 씁니다.
demo_documents = make_documents(demo_records)
practice_documents = make_documents(practice_records)
demo_by_id = {record["doc_id"]: record for record in demo_records}
practice_by_id = {record["doc_id"]: record for record in practice_records}

print(demo_documents[0])

## 1. Fixed: 길이와 겹침을 직접 정하기

**Fixed 청킹은 원문을 미리 정한 길이 기준으로 나누는 방식**입니다. 문장이나 주제가 끝났는지와 관계없이 정한 길이에 도달하면 자릅니다. 길이는 글자 수나 토큰 수로 정할 수 있으며, 이번 실습은 **공백을 포함한 글자 수**를 사용합니다. 마지막 청크는 남은 본문만 담으므로 더 짧을 수 있습니다.

여기서 Fixed는 **분할 전략의 이름**이고, `RecursiveCharacterTextSplitter`는 그 전략을 구현할 때 사용하는 **클래스 이름**입니다. 이 클래스에 `separators=[""]`를 주면 문단·줄바꿈·공백을 경계 후보로 쓰지 않고 문자 단위로 나누므로, 글자 수 기준 Fixed로 동작합니다.

- `chunk_size=80`: 청크 하나에 최대 80자를 담습니다. 경계를 눈으로 보려고 고른 작은 크기입니다.
- `chunk_overlap=20`: 앞 청크의 마지막 20자를 다음 청크에도 담습니다. 청크 크기가 100자로 늘어나는 것은 아닙니다.
- `length_function=len`: 길이를 파이썬 문자열의 글자 수로 셉니다. 토큰 수를 세는 설정과 구분하세요.

같은 클래스에서 `separators`를 생략하면 기본 Recursive 방식으로 문단·줄바꿈·공백 경계를 우선합니다. 아래에서 두 설정의 차이를 비교합니다. [LangChain 공식 문서](https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter)

<img style="background:#fff; max-width:100%; height:auto;" src="images/01_fixed_recursive.png" width="960" alt="문자 수로 자르는 Fixed와 구분자 경계를 우선하는 Recursive 비교" />

#### Fixed: 80자·겹침 0과 20을 같은 원문에서 비교하기

In [ ]:
# 글자 수와 구분자 기준으로 본문을 나누는 분할기입니다.
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 첫 원문 하나를 사용하고, 청크 크기는 80자로 고정한 채 겹침만 바꿉니다.
for overlap in [0, 20]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=80,  # 청크 하나의 최대 글자 수
        chunk_overlap=overlap,  # 다음 청크에 다시 담을 글자 수
        separators=[""],  # 문단·문장 경계를 사용하지 않고 글자 수로 자르는 Fixed 설정입니다.
        length_function=len,  # 공백과 줄바꿈도 글자 수에 포함합니다.
        strip_whitespace=False,  # 청크 앞뒤의 공백·줄바꿈을 유지합니다.
    )
    parts = splitter.split_documents(demo_documents[:1])

    # 겹침 설정별로 구역을 나누고, 각 청크의 길이와 본문을 따로 보여줍니다.
    print()
    print("=" * 60)
    print(f"겹침: {overlap}자")
    print(f"청크 수: {len(parts)}개")
    print("-" * 60)

    for chunk_number, part in enumerate(parts, start=1):
        print()
        print(f"[청크 {chunk_number}] 글자 수: {len(part.page_content)}자")
        print(repr(part.page_content))  # 앞뒤 공백과 줄바꿈도 보이도록 출력합니다.

이 예제의 원문은 **127자**입니다. 위치는 0부터 세고 끝 위치는 포함하지 않습니다. 예를 들어 `[0:80]`은 원문의 처음 80자입니다.

| 설정 | 첫째 청크 | 둘째 청크 | 두 청크에 함께 담긴 구간 |
|---|---|---|---|
| 겹침 0자 | `[0:80]`, 80자 | `[80:127]`, 47자 | 없음 |
| 겹침 20자 | `[0:80]`, 80자 | `[60:127]`, 67자 | `[60:80]`, 20자 |

겹침 0에서는 “다른 이용자의 예약이 없으면 한”과 “번에 한해 일주일 연장할 수 있습니다”가 서로 다른 청크에 놓입니다. 둘째 청크만 검색하면 **예약이 없어야 한다는 조건**을 놓칠 수 있습니다. 겹침 20에서는 둘째 청크가 60부터 시작해 이 문장을 함께 담습니다.

다만 둘째 청크는 앞 문장의 끝인 “다.”부터 시작합니다. **겹침은 경계 주변 내용을 다시 담는 설정이며, 문장이 온전히 들어가도록 판단하는 기능은 아닙니다.** 이 예제에서는 연장 조건이 보존되지만, 긴 문장에는 더 큰 겹침이 필요할 수 있습니다. 겹친 내용은 중복 저장되므로 입력량도 늘어납니다.

#### 같은 원문에 기본 Recursive 적용하기(80자·겹침 20자)

In [ ]:
# separators를 생략하면 문단·줄바꿈·공백·문자 순서의 기본값을 사용합니다.
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=80,
    chunk_overlap=20,  # Recursive는 경계에 맞춰 최대 20자까지 겹칩니다.
    strip_whitespace=False,
)
recursive_chunks = recursive_splitter.split_documents(demo_documents[:1])

for chunk in recursive_chunks:
    print("글자 수:", len(chunk.page_content))
    print(repr(chunk.page_content))

기본 Recursive는 문단 → 줄바꿈 → 공백 → 문자 순서로 더 작은 경계를 찾으며, 최대 80자 안에서 청크를 만듭니다. 이 원문에는 줄바꿈이 없어 공백 경계가 사용됩니다. 따라서 청크마다 길이가 달라지고 겹침도 정확히 20자가 아닐 수 있습니다. 문장 의미를 분석하는 방식은 아닙니다.

Fixed와 크기·겹침 설정은 같지만 **자를 위치를 고르는 기준**이 달라졌습니다. 두 방식의 둘째 청크 첫 부분과 겹친 내용, 문장 중간이 잘린 부분을 비교하세요.

### 🖐️ 함께 따라하기: 긴 인사 매뉴얼에 Fixed 적용

`practice_documents` 네 편 전체를 600자·겹침 100의 문자 단위로 나눠 `practice_fixed`에 담으세요. `separators=[""]`, `strip_whitespace=False`를 사용합니다. 이후 평가에서 원문 근거와 대조할 수 있도록 `add_start_index=True`로 시작 위치도 기록해 둡니다. 지금은 첫 3개 청크의 글자 수와 본문을 출력해 분할 결과를 읽으세요.

**확인 기준**: 첫 3개 청크는 각각 600자이며, 앞 청크의 마지막 100자가 다음 청크의 처음 100자와 같습니다. 문장이나 조건이 중간에서 잘렸는지 읽으세요.

In [ ]:
# 1) practice_documents를 600자·겹침 100으로 나눠 practice_fixed를 만드세요.
# 2) 첫 3개 청크의 글자 수와 본문을 출력하고 겹친 내용을 비교하세요.
# 여기에 코드를 작성하세요

### ✅ 바로 확인 퀴즈

**1. 600자·겹침 100이면 두 번째 청크는 긴 본문의 몇 번째 위치에서 시작하나요?**

<details><summary>정답 보기</summary><p>0부터 세어 500입니다. 첫 청크의 마지막 100자를 다시 포함합니다.</p></details>

**2. 줄바꿈이 많은 문서라면 기본 Recursive와 Fixed의 경계는 어떻게 달라질까요?**

<details><summary>정답 보기</summary><p>기본 Recursive는 문단·줄바꿈에서 먼저 자르려 해 경계가 문단에 맞춰지고, Fixed는 줄바꿈과 상관없이 정한 글자 수에서 자릅니다.</p></details>

## 2. Semantic: 이웃 문장의 의미가 달라지는 곳에서 나누기

Semantic 청킹은 **이웃 문장 주변의 의미 차이**로 경계를 고릅니다. `SemanticSplitterNodeParser`는 문장 주변 문맥을 임베딩하고 **코사인 거리(1 − 코사인 유사도)** 를 구합니다. 거리가 클수록 문맥이 더 다르다는 신호로 봅니다.

`buffer_size=1`은 각 문장에 존재하는 앞뒤 문장을 최대 1개씩 붙이는 설정입니다. 문장 2와 3 사이를 판단한다면 **1·2·3 문맥과 2·3·4 문맥**을 비교합니다. 이 묶음은 경계 판단용이며, 최종 청크의 겹침 설정은 아닙니다.

#### 거리 백분위란 무엇일까요?

**거리 백분위는 한 원문에서 구한 이웃 문맥의 거리들을 정렬했을 때, 지정한 백분율 위치에 해당하는 거리값**입니다. 기본 계산은 작은 값과 큰 값의 위치를 0%와 100%로 두고 정렬한 값들을 같은 간격으로 배치하며, 지정 위치가 두 값 사이면 그 사이 값을 보간합니다. [백분위 계산 기준](https://numpy.org/doc/stable/reference/generated/numpy.percentile.html)

`breakpoint_percentile_threshold=80`이면 그 원문의 **80백분위 거리값을 기준으로 삼아, 그 값보다 큰 곳에서 자릅니다.** 기준과 같은 값은 자르지 않습니다. 80은 유사도 80%, 거리 0.8, 원문의 80% 지점이 아닙니다. [파서의 거리·경계 계산](https://github.com/run-llama/llama_index/blob/main/llama-index-core/llama_index/core/node_parser/text/semantic_splitter.py)

#### 예: 거리 기준을 정하고 경계를 고릅니다

아래는 **설명용 가상 거리**이며 실제 임베딩 결과가 아닙니다. 각 값은 해당 문장 주변의 문맥을 반영합니다.

| 원문의 경계 후보 | 이웃 문맥의 거리 | 80백분위 기준 적용 |
|---|---:|---|
| 문장 1과 2 사이 | 0.20 | 유지 |
| 문장 2와 3 사이 | 0.70 | 자름 |
| 문장 3과 4 사이 | 0.25 | 유지 |

정렬한 값 **0.20, 0.25, 0.70**을 각각 **0·50·100백분위**에 놓습니다. 80백분위는 50에서 100으로 가는 구간의 **(80 − 50) ÷ (100 − 50) = 0.6**, 즉 60% 지점입니다.

따라서 50백분위 값 0.25에서 다음 값 0.70까지의 차이 중 60%만큼 더합니다. 이것이 **선형 보간**입니다.

$$
0.25 + 0.6 × (0.70 - 0.25) = 0.25 + 0.27 = 0.52
$$

기준값 0.52를 초과하는 0.70만 경계가 되어 **문장 1·2 / 문장 3·4**로 나뉩니다. 정렬은 기준값을 구할 때만 쓰며 원문의 순서를 바꾸지 않습니다.

<img style="background:#fff; max-width:100%; height:auto;" src="images/01_semantic.png" width="960" alt="설명용 거리 0.20·0.70·0.25에서 80백분위 0.52를 초과하는 문장 2와 3 사이를 분할" />

- **50 → 90으로 높이면**: 같은 거리 목록에서 기준값이 높아지거나 같아져, 경계가 줄거나 같습니다.
- **원문이 바뀌면**: 거리 분포도 달라져 같은 80백분위여도 실제 기준값은 달라집니다.
- **거리값이 반복되거나 적으면**: 서로 다른 백분위의 청크 수가 같을 수 있습니다. 정확히 상위 20% 개수를 자른다는 뜻은 아닙니다.

실습에서는 파서가 반환한 **Node(청크를 담는 객체)** 의 `text`와 글자 수를 읽습니다. `LangchainEmbedding`은 앞서 만든 임베딩 모델을 파서에 연결합니다.

#### 공식 파서와 기존 임베딩 연결 준비

In [ ]:
# 의미 경계 파서와 기존 임베딩 모델을 연결할 도구만 가져옵니다.
from llama_index.core import Document as IndexDocument
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.embeddings.langchain import LangchainEmbedding

# 이미 만든 OpenAI 임베딩을 그대로 연결합니다. 새 모델을 만들지 않습니다.
index_embedding = LangchainEmbedding(embedding_model)

In [ ]:
def to_index_documents(documents):
    """LangChain Document를 LlamaIndex 파서가 읽는 Document로 바꿉니다."""
    # 본문·metadata를 유지하면서 파서가 사용하는 Document 형식으로 바꿉니다.
    result = [IndexDocument.from_langchain_format(doc) for doc in documents]
    for doc in result:
        # 분할 노드의 ref_doc_id로 청킹 전 원문을 찾을 수 있도록 ID를 맞춥니다.
        doc.id_ = doc.metadata["source_id"]
        # metadata는 보관하되 모델 입력에서 빼서 긴 URL 등이 분할·의미 비교에 섞이지 않게 합니다.
        doc.excluded_embed_metadata_keys = list(doc.metadata)
        doc.excluded_llm_metadata_keys = list(doc.metadata)

    return result

#### 도서관 안내 네 편을 이어 붙인 원문 만들기

In [ ]:
# 서로 다른 주제가 들어 있는 원문을 만들어 의미 경계가 어디에 생기는지 봅니다.
library_record = {
    "doc_id": "lib_all",
    "title": "도서관 안내 전체",
    "url": "",
    "text": " ".join(record["text"] for record in demo_records),
}
library_documents = make_documents([library_record])

# 파서가 받는 형식으로 바꿉니다. 본문은 그대로이며 임베딩을 호출하지 않습니다.
library_index_documents = to_index_documents(library_documents)

#### Semantic: 거리 백분위 50과 90으로 나누기

In [ ]:
# 입력과 임베딩 모델은 같게 두고, 경계를 고르는 백분위만 비교합니다.
demo_sem_nodes = {}
for percentile in [50, 90]:
    semantic_parser = SemanticSplitterNodeParser(
        buffer_size=1,  # 각 문장의 앞뒤 최대 1문장씩을 문맥에 포함합니다.
        breakpoint_percentile_threshold=percentile,  # 이 백분위의 거리값을 초과하면 자릅니다.
        embed_model=index_embedding,  # 같은 임베딩 모델로 문맥 사이 거리를 구합니다.
    )

    # 이 호출에서 문맥을 임베딩하고 청크를 만듭니다. 반복할 때마다 임베딩을 요청합니다.
    demo_sem_nodes[percentile] = semantic_parser.get_nodes_from_documents(library_index_documents)

#### 백분위별 청크 수와 본문 비교하기

In [ ]:
# 백분위별로 결과를 구분하고, 각 청크의 첫 문장과 마지막 문장을 읽습니다.
for percentile, nodes in demo_sem_nodes.items():
    print()
    print("=" * 60)
    print(f"거리 기준: {percentile}백분위")
    print(f"청크 수: {len(nodes)}개")

    for chunk_number, node in enumerate(nodes, start=1):
        print()
        print(f"[청크 {chunk_number}] 글자 수: {len(node.text)}자")
        print(node.text)  # Node의 text가 파서가 나눈 청크 본문입니다.

같은 원문에서 50백분위와 90백분위의 청크 수와 본문을 비교하세요. 청크가 많아졌는지만 보지 말고, 주제가 바뀌는 곳에서 나뉘었는지와 조건·예외가 함께 남았는지 읽습니다.

### 🖐️ 함께 따라하기: 여러 페이지 본문의 의미 경계

`practice_documents`에서 `source_id`가 `hr_flexible_intro`인 원문 전체를 골라 백분위 50과 90으로 나누세요. PDF 9~20쪽의 유연근무 개념·도입 본문을 사용합니다. 같은 입력에서 백분위만 바꿔야 경계 선택의 차이를 비교할 수 있습니다.

- **파서 설정**: `buffer_size=1`, `breakpoint_percentile_threshold`는 50과 90, `embed_model=index_embedding`
- **입력 형식**: `to_index_documents`로 원문을 파서용 문서로 변환
- **결과**: `practice_sem_nodes` 딕셔너리에 백분위를 키로, 반환된 Node 목록을 값으로 저장
- **관찰**: 백분위별 전체 청크 수와 첫 3개 청크의 번호·글자 수·본문 출력

**확인 기준**: 같은 거리 분포에서는 50백분위 쪽 청크가 같거나 많습니다. 청크가 나뉜 부분을 읽고 조건·예외가 분리됐는지 확인하세요. 실제 청크 수와 경계 위치를 고정된 정답으로 두지는 않습니다.

In [ ]:
# 1) hr_flexible_intro 전체를 파서용 문서로 바꾸고 백분위 50과 90으로 나누세요.
# 2) 반환된 Node 목록을 practice_sem_nodes에 저장하세요.
# 3) 백분위별 청크 수와 첫 3개 청크의 글자 수·본문을 비교하세요.
# 여기에 코드를 작성하세요

### ✅ 바로 확인 퀴즈

**1. 거리의 80백분위를 기준으로 쓴다는 것은 유사도가 80%라는 뜻인가요?**

<details><summary>정답 보기</summary><p>아닙니다. 해당 원문의 이웃 문맥 거리들을 정렬해 80% 위치의 기준값을 정하고, 그 값보다 거리가 큰 곳을 자른다는 뜻입니다.</p></details>

**2. 90백분위로 높였는데 50백분위와 청크 수가 같을 수도 있나요?**

<details><summary>정답 보기</summary><p>네. 거리 표본이 작거나 값이 반복되면 기준을 넘는 경계 후보가 같을 수 있습니다. 높은 백분위가 반드시 더 적은 청크 수를 보장하지는 않습니다.</p></details>

**3. Semantic 분할도 GPT로 문장을 요약하나요?**

<details><summary>정답 보기</summary><p>아닙니다. 임베딩으로 경계를 고르고 원문을 나눕니다. 요약은 교안 02의 RAPTOR에서 다룹니다.</p></details>

## 3. Parent-Child: 자식으로 찾고 부모로 돌려주기

“일주일 연장”에 맞는 짧은 조각을 찾아도, 답에는 “다른 이용자의 예약이 없으면 한 번에 한해”라는 조건이 필요합니다. Parent-Child는 **작은 자식으로 검색**하고 **큰 부모를 돌려줍니다**.

| 객체 | 저장하는 것 | 하는 일 |
|---|---|---|
| Chroma | 자식 청크와 임베딩 | 질문으로 자식 검색 |
| InMemoryStore | 부모 Document | 부모 ID로 원문 조회 |
| ParentDocumentRetriever | 두 저장소의 연결 | 자식 검색 후 부모 반환 |

**`search_kwargs={"k": 3}`은 질문마다 검색할 자식 청크 수를 3개로 정합니다.** 질문과 가까운 자식 3개를 찾은 뒤, 각 자식의 `parent_id`로 부모 본문을 가져옵니다. 자식 3개의 부모 ID가 A·A·B이면 중복을 빼고 **부모 A·B 두 개**를 반환합니다.

짧은 도서관 안내는 한 편 전체를 부모로 쓰고, 긴 PDF는 먼저 큰 부모 청크로 나눕니다. [공식 API](https://reference.langchain.com/python/langchain-classic/retrievers/parent_document_retriever/ParentDocumentRetriever)

<img style="background:#fff; max-width:100%; height:auto;" src="images/01_parent_child.png" width="960" alt="자식 청크를 Chroma로 검색하고 부모 ID로 원문을 반환하는 Parent-Child" />

#### 도서관 자식·부모 저장소 연결하기

In [ ]:
# 자식은 Chroma로 검색하고, 부모는 메모리 저장소에서 ID로 꺼냅니다.
from langchain_chroma import Chroma
from langchain_core.stores import InMemoryStore
from langchain_classic.retrievers import ParentDocumentRetriever

# Chroma에는 검색용 자식, docstore에는 반환할 부모 원문을 보관합니다.
demo_pc_store = Chroma(collection_name="day46_demo_parent", embedding_function=embedding_model)
# 재실행할 때 이 수업용 컬렉션에 이전 자식이 중복 적재되지 않도록 비웁니다.
demo_pc_store.reset_collection()
demo_pc_docstore = InMemoryStore()  # 부모 본문은 임베딩하지 않고 메모리에 보관합니다.

demo_pc_retriever = ParentDocumentRetriever(
    vectorstore=demo_pc_store,  # 자식 청크의 벡터를 비교해 검색합니다.
    docstore=demo_pc_docstore,  # 찾은 자식의 parent_id로 부모를 꺼냅니다.
    child_splitter=RecursiveCharacterTextSplitter(
        chunk_size=80,
        chunk_overlap=20,
        add_start_index=True,  # 자식의 시작 위치는 각 부모 본문을 기준으로 기록합니다.
    ),
    id_key="parent_id",  # 자식 metadata에서 부모 ID를 담을 키 이름입니다.
    search_kwargs={"k": 3},  # 질문마다 검색할 자식 청크 수
)

# 이 호출에 넣는 Document가 부모입니다. 검색용 자식은 child_splitter가 나눕니다.
demo_pc_retriever.add_documents(
    # 아래 ID가 자식의 parent_id에 기록되고 docstore에서 부모를 찾는 키가 됩니다.
    demo_documents,
    ids=[doc.metadata["source_id"] for doc in demo_documents],
)

#### 같은 질문의 자식과 반환 부모를 각각 보기

In [ ]:
question = "도서 대출을 연장하려면 어떤 조건이 필요한가요?"

# 관찰을 위해 자식 검색과 부모 반환을 각각 호출합니다. 실제로는 invoke 한 번으로 부모를 얻습니다.
child_hits = demo_pc_store.similarity_search(question, k=3)
parent_hits = demo_pc_retriever.invoke(question)

# 서로 다른 자식이라도 parent_id가 같으면 같은 부모로 연결됩니다.
for child in child_hits:
    print("검색 자식 / 부모 ID:", child.metadata["parent_id"], child.page_content)

for parent in parent_hits:
    print("반환 부모:", parent.metadata["title"], parent.page_content)

`reset_collection()`은 이름을 준 컬렉션을 통째로 비웁니다. 지워도 되는 수업용 컬렉션 이름에만 쓰세요.

긴 본문 전체를 부모로 반환하면 질문 하나에 여러 페이지가 따라올 수 있습니다. PDF는 먼저 Recursive로 **부모 1,500자·겹침 200자**를 만들고, 각 부모에서 **자식 300자·겹침 50자**를 만듭니다. `source_id`는 긴 본문 ID, `parent_id`는 부모 저장소의 개별 청크 ID입니다. 부모의 문자 위치는 긴 본문 기준으로 기록합니다.

#### 긴 PDF를 반환용 부모와 검색용 자식으로 나누기

In [ ]:
# 긴 본문을 먼저 반환용 부모 청크로 나눕니다. 위치는 긴 본문 기준입니다.
practice_pc_parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
    strip_whitespace=False,
    add_start_index=True,
)
practice_pc_parent_documents = practice_pc_parent_splitter.split_documents(practice_documents)

for parent in practice_pc_parent_documents:
    # 부모가 원문의 어느 구간인지 기록해 검색 후 PDF 출처를 찾습니다.
    parent.metadata["end_index"] = parent.metadata["start_index"] + len(parent.page_content)

# 한 원문에 부모가 여러 개이므로 원문 ID와 시작 위치를 함께 키로 사용합니다.
practice_pc_parent_ids = [
    f"{doc.metadata['source_id']}:parent:{doc.metadata['start_index']}"
    for doc in practice_pc_parent_documents
]

# Chroma에는 검색용 자식, docstore에는 반환할 부모 원문을 보관합니다.
practice_pc_store = Chroma(collection_name="day46_practice_parent", embedding_function=embedding_model)
# 재실행할 때 이 수업용 컬렉션에 이전 자식이 중복 적재되지 않도록 비웁니다.
practice_pc_store.reset_collection()
practice_pc_docstore = InMemoryStore()  # 부모 본문은 임베딩하지 않고 메모리에 보관합니다.

practice_pc_retriever = ParentDocumentRetriever(
    vectorstore=practice_pc_store,
    docstore=practice_pc_docstore,
    child_splitter=RecursiveCharacterTextSplitter(
        chunk_size=300,
        chunk_overlap=50,
        add_start_index=True,
    ),
    id_key="parent_id",
    search_kwargs={"k": 3},
)

# 이 호출에 넣는 Document가 부모입니다. 검색용 자식은 child_splitter가 나눕니다.
practice_pc_retriever.add_documents(
    # 아래 ID가 자식의 parent_id에 기록되고 docstore에서 부모를 찾는 키가 됩니다.
    practice_pc_parent_documents,
    ids=practice_pc_parent_ids,
)

#### 검색 결과에 PDF 출처 표시하기

이제 검색된 부모 청크를 원본 PDF에서 찾아봅니다. `add_pdf_pages`는 청크의 원문 문자 구간과 `page_spans`를 대조해 PDF 시작·끝 페이지를 붙이는 제공 함수입니다. 청크가 여러 페이지에 걸치면 그 범위를 표시합니다. 본문이나 청크 경계를 바꾸지는 않습니다. [util.py](util.py)에서 가져와 사용합니다.

In [ ]:
# 원문 문자 구간을 PDF 페이지 범위와 연결하는 지원 함수입니다.
from util import add_pdf_pages

### 🖐️ 함께 따라하기: PDF 부모 청크의 범위와 원문 확인

방금 만든 `practice_pc_retriever`로 “유연근무 신청을 반려할 때 어떤 절차가 필요한가요?”를 검색해 `practice_parents`에 담으세요. `add_pdf_pages`에 부모 목록과 `practice_by_id`를 전달하고 각 부모의 글자 수·PDF 페이지 범위·본문을 출력하세요.

**확인 기준**: 부모는 3개 이하이고 각 부모는 1,500자 이하입니다. 부모 본문은 같은 `source_id`의 긴 본문에서 `start_index:end_index`로 꺼낸 문자열과 같습니다. 입력 본문 전체가 반환되는지, 큰 청크만 반환되는지 구분하세요.

In [ ]:
# 1) 신청 반려 질문으로 검색한 부모를 practice_parents에 담으세요.
# 2) add_pdf_pages로 출처를 붙이고 글자 수, 페이지 범위와 원문을 출력하세요.
# 여기에 코드를 작성하세요

### ✅ 바로 확인 퀴즈

**1. 부모 원문 전체를 임베딩해 검색해도 Parent-Child인가요?**

<details><summary>정답 보기</summary><p>아닙니다. Parent-Child는 작은 자식으로 순위를 정하고 부모는 돌려주기만 합니다.</p></details>

**2. `k=3`인데 부모가 2개만 반환됐다면 무슨 뜻인가요?**

<details><summary>정답 보기</summary><p>검색된 자식 3개 중 2개가 같은 부모에서 나왔다는 뜻입니다.</p></details>

## 4. Sentence Window: 검색한 문장의 앞뒤를 붙이기

`SentenceWindowNodeParser`는 원문을 문장으로 나누고 앞뒤 문장을 `metadata["window"]`에 담아 둡니다. Chroma에는 **한 문장만** 임베딩해 검색하고, 돌려줄 때 그 문장의 `window`로 바꿉니다. `window_size=1`은 검색 문장에 **존재하는 앞뒤 문장을 최대 하나씩** 붙이는 설정입니다. 문맥은 검색 뒤에 붙이므로 이미 정한 검색 순위는 그대로입니다.

<img style="background:#fff; max-width:100%; height:auto;" src="images/01_sentence_window.png" width="960" alt="문장으로 검색한 뒤 metadata의 앞뒤 문장을 답변 문맥으로 반환하는 Sentence Window" />

`chunk_overlap`은 저장 전에 청크를 겹치게 하고, Window는 검색 뒤에 돌려줄 문맥을 넓힙니다. [SentenceWindow 공식 소스](https://github.com/run-llama/llama_index/blob/main/llama-index-core/llama_index/core/node_parser/text/sentence_window.py)

#### 검색에 사용할 문서 변환과 원문 위치 연결

Chroma에 넣으려면 파서의 Node를 LangChain Document로 바꿉니다. 이후 PDF 출처 표시와 골든셋 평가에 사용할 원문 위치도 함께 연결합니다. 두 보조 함수는 [util.py](util.py)에서 가져옵니다. 내부 구현은 실습 범위에 포함하지 않습니다.

`nodes_to_documents`는 형식을 바꾸고, `restore_positions`는 본문과 원문 위치를 맞춥니다. 초기 파서 결과 전체는 원문 순서이므로 `sequential=True`를 쓰고, 검색 결과는 순위대로 섞이므로 기본값을 사용합니다.

In [ ]:
# 파서 결과를 Document로 바꾸고 원문 위치를 연결하는 지원 함수입니다.
from util import nodes_to_documents, restore_positions

#### 문장과 주변 문맥을 함께 준비하기

In [ ]:
# 한 문장을 검색 단위로, 앞뒤 문장을 반환 문맥으로 담는 파서입니다.
from llama_index.core.node_parser import SentenceWindowNodeParser

# 검색 본문은 한 문장, metadata["window"]는 앞뒤 최대 1문장을 포함한 문맥입니다.
window_parser = SentenceWindowNodeParser.from_defaults(window_size=1)  # 앞뒤 각각의 문장 수입니다.
window_nodes = window_parser.get_nodes_from_documents(to_index_documents(demo_documents))

# 한 문장인 검색 본문에 원문 위치를 붙이며, 확장 문맥은 window metadata에 그대로 둡니다.
demo_sentences = restore_positions(nodes_to_documents(window_nodes), demo_by_id, sequential=True)

#### 검색할 문장만 Chroma에 넣기

In [ ]:
# 문서 적재와 질문 검색에 같은 임베딩 설정을 사용합니다.
# add_documents가 넣을 문서 전체의 임베딩을 요청합니다. 돌려받는 ID 목록 대신 개수만 확인합니다.
demo_sentence_store = Chroma(collection_name="day46_demo_sentences", embedding_function=embedding_model)
demo_sentence_store.reset_collection()  # 다시 실행해도 같은 문서가 두 번 쌓이지 않도록 비웁니다.

added_ids = demo_sentence_store.add_documents(demo_sentences)
print("day46_demo_sentences 적재 수:", len(added_ids))

#### 질문으로 문장을 검색하기

In [ ]:
# window는 metadata에 있고, 검색 대상 본문은 한 문장입니다.
sentence_hits = demo_sentence_store.similarity_search("예약 시간에 늦게 도착하면 어떻게 되나요?", k=2)

for hit in sentence_hits:
    print("검색 문장:", hit.page_content)

#### 검색 문장을 앞뒤 문맥으로 바꾸기

In [ ]:
# 검색된 문장 대신 미리 담아 둔 window를 돌려줍니다(다시 검색하지 않습니다).
# 원래 검색 결과가 바뀌지 않도록 metadata는 복사해서 씁니다.
window_contexts = [
    Document(page_content=hit.metadata["window"], metadata=dict(hit.metadata))
    for hit in sentence_hits
]
window_contexts = restore_positions(window_contexts, demo_by_id)

for context in window_contexts:
    print("주변 문맥:", context.page_content)

### 🖐️ 함께 따라하기: 신청·승인 절차의 주변 문맥

`practice_documents`에 `window_size=1` 파서를 적용한 `practice_sentences`를 `day46_practice_sentences` 컬렉션의 `practice_sentence_store`에 넣으세요. “유연근무 신청을 반려할 때 어떤 절차가 필요한가요?”를 k=2로 검색해 `practice_hits`에 담고, 그 `window`로 만든 `practice_windows`에 원문 위치를 붙여 출력하세요.

**확인 기준**: 적재 수는 `len(practice_sentences)`와 같고, `practice_windows`는 2개 이하(같은 구간은 하나로 합침)입니다. `add_pdf_pages`로 반환 문맥의 페이지 범위를 붙이세요. 기본 문장 파서는 PDF 목록·소제목을 완전한 한국어 문장으로 구분하지 않을 수 있으므로 검색 문장과 확장 문맥을 직접 읽습니다.

In [ ]:
# 1) window_size=1 파서로 practice_sentences를 만들어 practice_sentence_store에 넣으세요.
# 2) k=2로 검색한 practice_hits를 window로 바꾼 practice_windows를 출력하세요.
# 여기에 코드를 작성하세요

### ✅ 바로 확인 퀴즈

**1. window 문맥 전체를 `page_content`에 넣어 임베딩해도 Sentence Window인가요?**

<details><summary>정답 보기</summary><p>아닙니다. 검색 대상이 한 문장에서 확장 문맥으로 바뀌므로 검색 순위도 달라질 수 있습니다.</p></details>

**2. PDF 페이지가 바뀌면 Window도 반드시 끊기나요?**

<details><summary>정답 보기</summary><p>아닙니다. 여러 페이지를 한 본문으로 연결했으므로 같은 본문 안에서는 페이지를 넘어 앞뒤 문장을 붙일 수 있습니다. 서로 다른 입력 본문까지 확장하지는 않습니다.</p></details>

## 5. Auto-merging: 같은 부모의 자식이 많이 잡히면 부모로 합치기

`HierarchicalNodeParser`가 부모·자식 계층을 만들고, `AutoMergingRetriever`는 작은 자식(잎)으로 검색한 뒤 **한 부모의 자식이 충분히 모이면 그 부모 청크로 바꿉니다.** 조건은 `검색된 그 부모의 자식 수 / 그 부모의 전체 자식 수 > simple_ratio_thresh`입니다. 기준 0.5에서 자식 4개 중 2개는 그대로, 3개면 부모로 바뀝니다.

크기 단위는 글자가 아니라 **토큰**입니다. 같은 토큰 수라도 언어·문장에 따라 글자 수는 달라집니다. 출력에서 실제 반환 길이를 확인합니다.

<img style="background:#fff; max-width:100%; height:auto;" src="images/01_auto_merging.png" width="960" alt="같은 부모의 자식 검색 비율이 기준을 초과하면 계층 부모로 대체하는 Auto-merging" />

| 도구 | 입력 → 결과 |
|---|---|
| `HierarchicalNodeParser` | 원문 → 부모·자식 노드 |
| `StorageContext.docstore` | 전체 노드 보관 → ID로 부모 조회 |
| `VectorStoreIndex` | 잎 노드 → 검색용 메모리 인덱스(Chroma 대신) |
| `AutoMergingRetriever` | 잎 검색 결과 → 부모 또는 자식 |

시연은 부모 128·자식 32토큰입니다. 크기가 작아 `Metadata length (0) is close to chunk size` 안내가 찍히지만 결과에는 영향이 없습니다. [AutoMergingRetriever 공식 소스](https://github.com/run-llama/llama_index/blob/main/llama-index-core/llama_index/core/retrievers/auto_merging_retriever.py)

#### 도서관의 계층과 Auto-merging 검색기 만들기

In [ ]:
# 계층을 만들고, 검색된 잎을 조건에 따라 부모로 병합하는 도구입니다.
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.core.node_parser import HierarchicalNodeParser, get_leaf_nodes
from llama_index.core.retrievers import AutoMergingRetriever

# 크기는 토큰 수입니다. 부모(큰 청크)와 자식(작은 청크) 두 크기로 계층을 만듭니다.
demo_auto_parser = HierarchicalNodeParser.from_defaults(
    chunk_sizes=[128, 32],  # 큰 부모부터 작은 자식 순서이며 단위는 토큰입니다.
    chunk_overlap=0,  # 같은 부모의 자식들이 겹치지 않게 하여 원문 위치를 순서대로 연결합니다.
    include_prev_next_rel=False,  # 앞뒤 노드 연결을 만들지 않아 부모 병합만 일어나게 합니다.
)
demo_auto_nodes = demo_auto_parser.get_nodes_from_documents(to_index_documents(demo_documents))

demo_auto_storage = StorageContext.from_defaults()
# 부모를 찾아 반환할 수 있도록 전체 노드는 docstore에, 검색용 잎만 인덱스에 둡니다.
demo_auto_storage.docstore.add_documents(demo_auto_nodes)

demo_auto_index = VectorStoreIndex(
    get_leaf_nodes(demo_auto_nodes),  # 가장 작은 잎만 임베딩해 질문으로 검색합니다.
    storage_context=demo_auto_storage,  # 병합할 부모를 같은 docstore에서 찾습니다.
    embed_model=index_embedding,  # 문서 적재와 질문 검색에 같은 임베딩 모델을 씁니다.
)

demo_auto_retriever = AutoMergingRetriever(
    demo_auto_index.as_retriever(similarity_top_k=3),  # 병합 전 검색할 잎의 수입니다.
    demo_auto_storage,
    simple_ratio_thresh=0.5,  # 검색된 자식 수 / 부모의 전체 자식 수 > 0.5일 때 병합합니다.
)

#### 검색된 자식과 최종 반환 범위 읽기

In [ ]:
auto_question = "책의 대출 기간과 연장 조건은?"

# 원리를 관찰하기 위해 병합 전 검색기와 병합 검색기를 각각 호출합니다.
auto_child_hits = demo_auto_index.as_retriever(similarity_top_k=3).retrieve(auto_question)
auto_hits = demo_auto_retriever.retrieve(auto_question)

auto_contexts = restore_positions(
    nodes_to_documents([hit.node for hit in auto_hits], demo_auto_nodes, demo_by_id),
    demo_by_id,
)

for hit in auto_child_hits:
    parent_id = hit.node.parent_node.node_id  # 검색된 잎이 속한 부모를 확인합니다.
    parent_node = demo_auto_storage.docstore.get_document(parent_id)
    print("검색 자식 / 부모 ID:", parent_id, hit.node.text)
    print("이 부모의 전체 자식 수:", len(parent_node.child_nodes))

for context in auto_contexts:
    print("최종 반환:", context.metadata["title"], context.page_content)

같은 부모 ID가 나온 횟수를 그 부모의 전체 자식 수로 나누세요. 0.5를 넘은 부모만 최종 반환에서 부모 청크로 바뀝니다.

### 🖐️ 함께 따라하기: 긴 인사 매뉴얼로 Auto-merging 검색기 만들기

`practice_documents`로 부모 1,024·자식 256토큰(겹침 0, `include_prev_next_rel=False`) 계층을 만들고 잎 검색 수 3·기준 0.5의 `practice_auto_retriever`를 구성하세요. “유연근무 신청을 반려할 때 어떤 절차가 필요한가요?”로 검색해 원문 위치를 붙인 `practice_merged`를 만들고 페이지 범위를 붙이세요. `nodes_to_documents`의 두 번째 인자로 전체 계층 `practice_auto_nodes`, 세 번째 인자로 청킹 전 원문 `practice_by_id`를 전달합니다.

**확인 기준**: 반환 개수는 3개 이하입니다. 반환 노드가 전체 계층의 잎인지 부모인지 ID로 확인하고, 각 문맥의 실제 글자 수와 PDF 페이지 범위를 읽으세요. 검색된 형제 비율에 따라 병합이 일어나지 않을 수 있습니다.

In [ ]:
# 1) 1,024·256토큰 계층으로 practice_auto_retriever를 만드세요.
# 2) 전체 계층 목록으로 위치를 연결한 practice_merged에 페이지 범위를 붙이고 반환 종류와 길이를 출력하세요.
# 여기에 코드를 작성하세요

### ✅ 바로 확인 퀴즈

**1. `k=3`, 기준 0.5에서 자식 5개인 부모가 병합되려면 자식이 몇 개 검색돼야 하나요?**

<details><summary>정답 보기</summary><p>3개 모두입니다. 2/5=0.4는 기준 이하, 3/5=0.6은 초과입니다.</p></details>

**2. Parent-Child와 Auto-merging은 언제 부모를 돌려주나요?**

<details><summary>정답 보기</summary><p>Parent-Child는 항상, Auto-merging은 같은 부모의 자식 비율이 기준을 넘을 때만 돌려줍니다.</p></details>

## 6. 같은 질문으로 검색 품질 비교하기

지표는 **근거 구간 Recall@K**입니다. 각 전략에서 K개를 검색한 뒤 반환한 문맥을 합쳐, 사람이 정한 근거 구간 각각을 **전부 덮었는지** 셉니다. 부모 중복 제거·병합이 있으면 반환 개수는 K보다 적을 수 있습니다. 두 청크가 나눠 덮어도 회수이고, 같은 근거를 두 번 덮어도 한 번만 셉니다.

<img style="background:#fff; max-width:100%; height:auto;" src="images/02_evidence_ranges.png" width="960" alt="두 청크가 함께 하나의 고정 근거 구간을 덮는 경우도 회수로 인정" />

질문과 근거를 미리 정한 평가 자료를 **골든셋**이라 합니다. 도서관 4문항으로 계산을 익힌 뒤 PDF 평가 자료 **82문항 중 본문별 2문항씩** 먼저 비교합니다. 전체 문항으로 넓힐 수 있습니다. `evidence`는 청킹 전 긴 본문의 근거 구간, `reference_answer`는 참고 답안이며 인덱스에는 넣지 않습니다. 지정한 근거 구간을 회수하는 지표이므로 같은 내용을 다른 위치에서 찾으면 점수에 반영되지 않을 수 있습니다.

원문·질문·근거·임베딩·K(=3, Parent-Child·Auto-merging은 자식 3개)는 모든 전략에 같게 두고, 설정은 앞 절을 이어 씁니다. **K가 같아도 반환량은 다르므로** `recall`과 `context_chars`를 함께 읽으세요. 이 표는 이 데이터와 설정의 결과이며 전략 자체의 우열을 뜻하지 않습니다.

#### 같은 질문과 고정 근거 읽기

In [ ]:
# 전략마다 같은 질문과 근거를 사용합니다. 원문 Document는 앞 절의 것을 이어 씁니다.
demo_questions = read_json("demo_questions.json")
practice_questions = read_json("practice_questions.json")

print("도서관 설명용:", len(demo_questions), "/ PDF 평가용:", len(practice_questions))

#### 평가에 사용할 청크의 끝 위치 기록하기

골든셋의 근거 구간과 검색 결과를 대조하려면 청크의 시작·끝 위치가 필요합니다. `set_end_offsets`는 분할기가 기록한 `start_index`에 청크의 글자 수를 더해 `end_index`를 기록합니다. 끝 위치는 포함하지 않으며, 청크의 본문은 그대로입니다. 이 함수는 평가 준비에 사용하며 [util.py](util.py)에서 가져옵니다.

In [ ]:
# 시작 위치와 청크 길이로 끝 위치를 기록하는 지원 함수입니다.
from util import set_end_offsets

#### 여러 반환 구간을 합쳐 근거를 세는 함수

In [ ]:
def evidence_recall(contexts, evidence):
    """반환한 원문 구간들이 고정 근거를 얼마나 회수했는지 계산합니다.

    Args:
        contexts: source_id, start_index, end_index를 가진 원문 Document 리스트.
        evidence: doc_id, start, end를 가진 비어 있지 않은 근거 리스트.
    Returns:
        전체 위치가 회수된 근거 수 / 전체 근거 수. 답변 정확도는 아닙니다.
    """
    covered = {}
    for context in contexts:
        meta = context.metadata
        positions = covered.setdefault(meta["source_id"], set())
        # 출처별 문자 위치의 합집합이므로 겹친 청크가 근거 수를 늘리지 않습니다.
        positions.update(range(meta["start_index"], meta["end_index"]))

    found = 0
    for item in evidence:
        required = set(range(item["start"], item["end"]))
        # 다른 원문의 같은 위치는 인정하지 않으며, 근거 일부만 덮은 경우도 제외합니다.
        if required.issubset(covered.get(item["doc_id"], set())):
            found += 1

    return found / len(evidence)

Parent-Child·Window·Auto-merging은 앞 절의 검색기를 그대로 쓰고, Fixed와 Semantic만 비교용 인덱스를 새로 만듭니다. Semantic도 여기서는 Node를 Document로 바꾸고 `util.py`의 함수로 원문 위치를 붙여 채점합니다.

#### Fixed 청크와 인덱스 준비

In [ ]:
# 빈 구분자만 사용해 문단 경계 대신 문자 수로 자릅니다.
fixed_splitter = RecursiveCharacterTextSplitter(
    chunk_size=80,
    chunk_overlap=20,
    separators=[""],
    strip_whitespace=False,
    add_start_index=True,
)

# 분할기가 주는 시작 위치에 청크 길이를 더해 끝 위치도 기록합니다.
demo_fixed = set_end_offsets(fixed_splitter.split_documents(demo_documents))

# 문서 적재와 질문 검색에 같은 임베딩 설정을 사용합니다.
# add_documents가 넣을 문서 전체의 임베딩을 요청합니다. 돌려받는 ID 목록 대신 개수만 확인합니다.
fixed_store = Chroma(collection_name="day46_compare_fixed", embedding_function=embedding_model)
fixed_store.reset_collection()  # 다시 실행해도 같은 문서가 두 번 쌓이지 않도록 비웁니다.

added_ids = fixed_store.add_documents(demo_fixed)
print("day46_compare_fixed 적재 수:", len(added_ids))

#### SemanticSplitterNodeParser로 의미 청크 준비

In [ ]:
# 이웃 문맥의 코사인 거리 중 지정 백분위보다 큰 지점에서 나눕니다.
semantic_parser = SemanticSplitterNodeParser(
    buffer_size=1,
    breakpoint_percentile_threshold=80,
    embed_model=index_embedding,
)
semantic_nodes = semantic_parser.get_nodes_from_documents(to_index_documents(demo_documents))

# 초기 전체 결과는 원문 순서로 위치를 붙입니다. 검색 결과에는 sequential=True를 쓰지 않습니다.
demo_semantic = restore_positions(nodes_to_documents(semantic_nodes), demo_by_id, sequential=True)

# 문서 적재와 질문 검색에 같은 임베딩 설정을 사용합니다.
# add_documents가 넣을 문서 전체의 임베딩을 요청합니다. 돌려받는 ID 목록 대신 개수만 확인합니다.
semantic_store = Chroma(collection_name="day46_compare_semantic", embedding_function=embedding_model)
semantic_store.reset_collection()  # 다시 실행해도 같은 문서가 두 번 쌓이지 않도록 비웁니다.

added_ids = semantic_store.add_documents(demo_semantic)
print("day46_compare_semantic 적재 수:", len(added_ids))

#### 질문 하나의 근거와 검색 구간을 먼저 대조하기

In [ ]:
example_question = demo_questions[2]  # 근거가 두 개인 질문이라 부분 회수 계산을 볼 수 있습니다.
example_hits = fixed_store.similarity_search(example_question["question"], k=3)
print("질문:", example_question["question"])

# 고정 근거와 반환 구간의 위치는 같은 source_id 안에서만 비교합니다.
for evidence in example_question["evidence"]:
    recovered = evidence_recall(example_hits, [evidence]) == 1.0
    print("필요한 근거:", evidence["doc_id"], evidence["start"], evidence["end"], evidence["quote"])
    print("이 근거의 전체 구간 회수:", recovered)

for hit in example_hits:
    meta = hit.metadata
    print("반환 구간:", meta["source_id"], meta["start_index"], meta["end_index"], hit.page_content)

print("질문 전체 Recall:", evidence_recall(example_hits, example_question["evidence"]))

회수가 `True`인 근거 수를 2로 나눈 값이 `질문 전체 Recall`과 같은지 확인하세요. 아래에서는 이 계산을 모든 질문과 전략에 반복합니다.

#### 질문마다 검색하고 반환 문맥만 평가하기

In [ ]:
# 질문별 결과를 먼저 남겨야 평균에서 드러나지 않는 근거 누락도 찾을 수 있습니다.
rows = []
for item in demo_questions:
    question = item["question"]

    # 질문은 같게 두고, 각 전략이 실제로 반환한 문맥을 비교합니다.
    window_hits = demo_sentence_store.similarity_search(question, k=3)

    # 검색된 문장 대신 미리 담아 둔 window를 돌려줍니다(다시 검색하지 않습니다).
    # 원래 검색 결과가 바뀌지 않도록 metadata는 복사해서 씁니다.
    window_contexts = [
        Document(page_content=hit.metadata["window"], metadata=dict(hit.metadata))
        for hit in window_hits
    ]
    window_contexts = restore_positions(window_contexts, demo_by_id)

    # Auto-merging은 5절의 계층 잎 인덱스로 따로 검색합니다.
    auto_hits = demo_auto_retriever.retrieve(question)
    auto_contexts = restore_positions(
        nodes_to_documents([hit.node for hit in auto_hits], demo_auto_nodes, demo_by_id), demo_by_id
    )

    contexts_by_strategy = {
        "Fixed": fixed_store.similarity_search(question, k=3),
        "Semantic": semantic_store.similarity_search(question, k=3),
        "Parent-Child": demo_pc_retriever.invoke(question),
        "Sentence Window": window_contexts,
        "Auto-merging": auto_contexts,
    }

    for strategy, contexts in contexts_by_strategy.items():
        # 반환량은 부분 겹침도 합산합니다. Recall의 문자 구간 합집합과 목적이 다릅니다.
        rows.append({
            "question_id": item["question_id"],
            "strategy": strategy,
            "recall": evidence_recall(contexts, item["evidence"]),
            "context_chars": sum(len(doc.page_content) for doc in contexts),
        })

results = pd.DataFrame(rows)
display(results)

#### 전략별 평균 비교하기

In [ ]:
# 질문마다 같은 비중으로 Recall을 평균냅니다. 전체 근거 수를 합쳐 나눈 값은 아닙니다.
comparison = results.groupby("strategy")[["recall", "context_chars"]].mean()
display(comparison)

도서관 4문항은 계산 과정을 익히는 용도라 이 평균으로 전략을 고르지 않습니다. 전략별 누락은 아래 PDF 문항에서 확인하세요.

PDF 따라하기는 1절의 `practice_fixed`와 4절의 `practice_sentence_store`를 이어 씁니다. Window 문서를 다시 임베딩하지 않습니다. 아래 기본 선택은 본문별 첫 두 문항이며 전체 품질을 대표하는 통계 표본은 아닙니다. 전체 질문으로 바꾸면 두 전략의 질문 임베딩 요청도 문항 수에 비례해 늘어납니다.

#### 본문별 두 질문으로 비교 범위 정하기

In [ ]:
# 먼저 각 긴 본문에서 두 문항씩 비교합니다. 근거 내용으로 검색 결과를 고르지는 않습니다.
evaluation_questions = []
for source_id in practice_by_id:
    source_questions = [
        item for item in practice_questions
        if item["evidence"][0]["doc_id"] == source_id
    ]
    evaluation_questions.extend(source_questions[:2])  # 본문별 두 질문으로 먼저 비교합니다.

# 전체 평가가 필요하면 아래 줄의 주석을 해제합니다.
# evaluation_questions = practice_questions
print("평가 질문 수:", len(evaluation_questions))
display(pd.DataFrame(evaluation_questions)[["question_id", "question"]])

### 🖐️ 함께 따라하기: 긴 PDF 본문에서 Fixed와 Window 비교

`evaluation_questions`로 두 전략을 비교하세요. 두 전략은 같은 긴 본문 네 편을 입력받았습니다.

- **Fixed**: 1절의 `practice_fixed`(600자·겹침 100)에 `set_end_offsets`로 끝 위치를 기록한 뒤 `day46_practice_compare_fixed` 컬렉션(`practice_fixed_store`)에 적재
- **Window**: 4절의 `practice_sentence_store`를 그대로 재사용
- **검색·반환**: 질문마다 k=3. Window는 검색 문장의 `window`를 반환 본문으로 쓰고 `restore_positions`로 원문 위치를 붙임
- **결과**: 질문·전략마다 `question_id`, `strategy`, `recall`, `context_chars`를 `practice_rows`에 담고 전략별 평균 표 출력

**확인 기준**: `practice_rows`는 `2 * len(evaluation_questions)`행이고 평균 표는 두 줄입니다. 평균과 함께 질문별 표에서 회수율이 낮은 질문을 찾으세요. 표의 점수만으로 어떤 조건이 빠졌는지까지 판단할 수는 없습니다.

In [ ]:
# 1) set_end_offsets로 Fixed 청크의 끝 위치를 기록하고 practice_fixed_store에 적재하세요.
# 2) practice_sentence_store를 재사용해 evaluation_questions를 두 전략으로 검색하세요.
# 3) 질문별 결과를 practice_rows에 담고 질문별 표와 전략별 평균을 출력하세요.
# 여기에 코드를 작성하세요

### ✅ 바로 확인 퀴즈

**1. 근거가 위치 30~90이고 반환 문맥이 20~60, 55~100이면 회수인가요?**

<details><summary>정답 보기</summary><p>회수입니다. 두 구간을 합치면 20~100이라 근거 전체를 덮습니다.</p></details>

**2. 근거가 두 개인 질문에서 하나는 전부, 다른 하나는 절반만 덮었다면 Recall은 얼마인가요?**

<details><summary>정답 보기</summary><p>0.5입니다. 근거는 전부 덮어야 회수로 세므로 절반만 덮은 근거는 세지 않습니다.</p></details>

**3. 질문을 검색 전에 다시 쓰는 일과, 찾은 결과를 답변 전에 추려 내는 일은 각각 Advanced RAG의 어느 단계인가요?**

<details><summary>정답 보기</summary><p>앞은 검색 전(Pre-retrieval), 뒤는 검색 후(Post-retrieval)입니다. 오늘 바꾼 청킹·인덱스 구성은 검색 전 단계입니다.</p></details>

## 이번 강의 정리

| 전략 | 검색 단위 | 반환 단위 | 함께 볼 것 |
|---|---|---|---|
| Fixed | 정한 길이의 청크 | 같은 청크 | 크기·겹침 |
| Semantic | 의미로 묶은 청크 | 같은 청크 | 백분위·문장 수 |
| Parent-Child | 작은 자식 | 부모 원문 | 반환 글자 수 |
| Sentence Window | 문장 | 앞뒤 문장까지 | window_size |
| Auto-merging | 계층의 잎 | 부모 또는 잎 | 자식 비율·k |

Advanced RAG의 검색 전 단계 가운데 **인덱싱**을 바꾸고, **근거 구간 Recall과 반환 글자 수**를 함께 비교했습니다.

## ⏭️ 다음 시간 예고

교안 02: RAPTOR. 원문을 군집으로 묶고 GPT 요약을 계층으로 쌓아 검색합니다.